# DistilBERT - classic4

Fine-tuning de DistilBERT com divisão estratificada 70/10/20 e avaliação por acurácia, F1 macro e F1 micro.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

def localizar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados").exists() and (candidato / "04-classificacao-distilbert").exists():
            return candidato
    raise FileNotFoundError("Não foi possível localizar a raiz do repositório.")

ROOT = localizar_raiz()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)


In [ ]:
DATA_PATH = ROOT / "dados" / "classic4.csv"
df = pd.read_csv(DATA_PATH).dropna(subset=["text", "class"])[["text", "class"]]
df.head()


In [ ]:
classes = sorted(df["class"].unique())
label2id = {label: idx for idx, label in enumerate(classes)}
id2label = {idx: label for label, idx in label2id.items()}

df = df.copy()
df["label"] = df["class"].map(label2id)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42,
)
validation_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,
    stratify=temp_df["label"],
    random_state=42,
)

print("Treino:", len(train_df))
print("Validação:", len(validation_df))
print("Teste:", len(test_df))


In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(validation_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

exemplo = tokenizer(train_df.iloc[0]["text"], truncation=True)
print("Primeiros tokens:", tokenizer.convert_ids_to_tokens(exemplo["input_ids"][:30]))


In [ ]:
def tokenizar(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

encoded = dataset.map(tokenizar, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
encoded


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_micro": f1_score(labels, preds, average="micro"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(classes),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
training_args = TrainingArguments(
    output_dir="checkpoints/distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    push_to_hub=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
resultado_teste = trainer.predict(encoded["test"])
resultado_teste.metrics


In [ ]:
y_true = resultado_teste.label_ids
y_pred = np.argmax(resultado_teste.predictions, axis=-1)
cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(values_format=".2f", ax=ax, colorbar=False)
plt.xticks(rotation=90)
plt.title("Matriz de confusão normalizada")
plt.tight_layout()
plt.show()


## Interpretação

Use F1 macro para observar o equilíbrio de desempenho entre as classes e F1 micro para o desempenho agregado. A matriz de confusão ajuda a identificar pares de classes que o modelo ainda confunde com frequência.
